# Kafka Basics

A first look at Kafka. By the end of this notebook you will have:

1. Started a Kafka server
2. Created a topic
3. Sent messages to it
4. Read those messages back
5. Asked a few simple questions about them with Spark

That is all. No streaming queries, no time windows. When you are comfortable with this,
move on to `kafka_streaming_lab_advanced.ipynb`.

**How to run it.** Click into the first cell and press Shift and Enter together. That runs
the cell and moves to the next one. Keep going, top to bottom, in order.

The setup takes about 2 to 3 minutes. Everything after that is quick.

---

## What is Kafka, in one picture

Kafka is a **message queue**. One program puts messages in, another takes them out, and
Kafka holds them in between.

```
   your program            Kafka             another program
   ------------            -----             ---------------
   "alice bought"  ---->  [ message ]  ---->  reads it later
                          [ message ]
                          [ message ]
```

Three words you need:

| Word | Meaning |
|---|---|
| **Broker** | The Kafka server itself. It stores the messages. |
| **Topic** | A named mailbox inside the broker. You will make one called `clicks`. |
| **Message** | One piece of data. Ours will be a short line of text. |

**Why not just have the two programs talk directly?** Because then both have to be running
at the same time, and each has to know about the other. With Kafka in the middle, the
sender can send whether or not anyone is listening, and the messages wait until someone
reads them.

## Step 1. Setup

Four cells that install what you need. Run them in order and wait for each to finish.

You do not need to understand this code. It downloads software, and the comments explain
the choices in case you are curious later.

| Cell | Installs |
|---|---|
| 1 | Java, which Kafka and Spark are both written in |
| 2 | PySpark, and a Kafka library for Python |
| 3 | Kafka itself |
| 4 | The piece that lets Spark read from Kafka |

In [1]:
# --- Java ----------------------------------------------------------------
# Spark 4 needs Java 17 or newer. Colab sometimes already has it, so check before
# installing: the apt install is the slowest part of this cell when it is needed.
import os, subprocess, time, re

t0 = time.time()

def java_major(home=None):
    exe = (home + '/bin/java') if home else 'java'
    try:
        out = subprocess.run([exe, '-version'], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

JAVA_HOME = '/usr/lib/jvm/java-17-openjdk-amd64'

if java_major() >= 17:
    print('Java 17+ already here, nothing to install')
    JAVA_HOME = subprocess.run(
        ['bash', '-lc', 'dirname $(dirname $(readlink -f $(which java)))'],
        capture_output=True, text=True).stdout.strip()
elif java_major(JAVA_HOME) >= 17:
    print('Java 17 already installed at', JAVA_HOME)
else:
    print('installing Java 17 (JRE only, which is all Spark and Kafka need)...')
    # jre-headless, not jdk-headless: no compiler, so a smaller download.
    !apt-get install -y openjdk-17-jre-headless -qq > /dev/null 2>&1

os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = JAVA_HOME + '/bin:' + os.environ['PATH']
print('JAVA_HOME =', JAVA_HOME)
!java -version
print('--- %.1f seconds ---' % (time.time() - t0))

Java 17+ already here, nothing to install
JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
--- 0.3 seconds ---


In [2]:
# --- PySpark and a Kafka-python client ------------------------------------------
# kafka-python is a producer you drive from Python. Version 1 sent messages by
# launching kafka-console-producer.sh, which starts a whole JVM for every call:
# about 1 to 2 seconds per message. This sends 8 messages in about 0.15 seconds.
import time
t0 = time.time()

!pip install -q pyspark==4.0.1 kafka-python

import pyspark
print('pyspark     ', pyspark.__version__)
import kafka
print('kafka-python', kafka.__version__)
print('--- %.1f seconds ---' % (time.time() - t0))

pyspark      4.0.1
kafka-python 3.0.11
--- 6.1 seconds ---


In [3]:
# --- Apache Kafka ----------------------------------------------------------------
# The CDN at dlcdn.apache.org only carries current releases, and which release is
# current changes over the year. So ask it what it has rather than hard-coding a
# version that will quietly disappear. archive.apache.org has every version ever
# released but is throttled to about 120 KB/s, so it is the fallback, not the default.
import os, re, time, urllib.request

KAFKA_DIR = '/content/kafka'
t0 = time.time()

def newest_on_cdn():
    html = urllib.request.urlopen(
        'https://dlcdn.apache.org/kafka/', timeout=30).read().decode()
    vers = sorted({v.rstrip('/') for v in re.findall(r'\d+\.\d+\.\d+/', html)},
                  key=lambda s: [int(p) for p in s.split('.')])
    return vers[-1] if vers else None

if not os.path.exists(KAFKA_DIR + '/bin'):
    try:
        KAFKA_VERSION = newest_on_cdn()
        URL = 'https://dlcdn.apache.org/kafka/%s/kafka_2.13-%s.tgz' % (
            KAFKA_VERSION, KAFKA_VERSION)
        print('CDN has Kafka %s, downloading at CDN speed...' % KAFKA_VERSION)
    except Exception as e:
        KAFKA_VERSION = '3.9.1'
        URL = ('https://archive.apache.org/dist/kafka/%s/kafka_2.13-%s.tgz'
               % (KAFKA_VERSION, KAFKA_VERSION))
        print('CDN unreachable (%s). Falling back to the slow archive.' % e)

    !wget -q --show-progress -O /content/kafka.tgz {URL}
    !mkdir -p {KAFKA_DIR}
    !tar xzf /content/kafka.tgz -C {KAFKA_DIR} --strip-components=1
    !rm -f /content/kafka.tgz
else:
    print('Kafka already unpacked')

!{KAFKA_DIR}/bin/kafka-topics.sh --version
print('--- %.1f seconds ---' % (time.time() - t0))

Kafka already unpacked
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
4.3.1
--- 5.8 seconds ---


In [4]:
# --- The Spark Kafka connector -------------------------------------------
# Spark cannot read Kafka without four extra jars. The usual way is
# .config('spark.jars.packages', ...), which makes Spark run Ivy to resolve them
# from Maven on every start: often a few minutes in Colab, and it sometimes stalls.
# Downloading the four files straight into Spark's own jars folder takes about a
# second, and then the SparkSession starts with no network calls at all.
import os, time, pyspark
t0 = time.time()

JARS = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
MAVEN = 'https://repo1.maven.org/maven2'
SV = pyspark.__version__

# kafka-clients and commons-pool2 versions come from the connector's own pom for
# Spark 4.0.1. If you change the PySpark version, check that pom again.
WANTED = [
    ('%s/org/apache/spark/spark-sql-kafka-0-10_2.13/%s/'
     'spark-sql-kafka-0-10_2.13-%s.jar' % (MAVEN, SV, SV),
     'spark-sql-kafka-0-10_2.13-%s.jar' % SV),
    ('%s/org/apache/spark/spark-token-provider-kafka-0-10_2.13/%s/'
     'spark-token-provider-kafka-0-10_2.13-%s.jar' % (MAVEN, SV, SV),
     'spark-token-provider-kafka-0-10_2.13-%s.jar' % SV),
    ('%s/org/apache/kafka/kafka-clients/3.9.1/kafka-clients-3.9.1.jar' % MAVEN,
     'kafka-clients-3.9.1.jar'),
    ('%s/org/apache/commons/commons-pool2/2.12.0/commons-pool2-2.12.0.jar' % MAVEN,
     'commons-pool2-2.12.0.jar'),
]

for url, name in WANTED:
    dest = os.path.join(JARS, name)
    if not os.path.exists(dest):
        !wget -q -O {dest} {url}
    print('%7.1f KB  %s' % (os.path.getsize(dest) / 1024, name))

print('--- %.1f seconds ---' % (time.time() - t0))

  456.6 KB  spark-sql-kafka-0-10_2.13-4.0.1.jar
   63.5 KB  spark-token-provider-kafka-0-10_2.13-4.0.1.jar
 8999.2 KB  kafka-clients-3.9.1.jar
  146.5 KB  commons-pool2-2.12.0.jar
--- 0.0 seconds ---


If all four cells finished without a red error, you are ready. The slowest is cell 2,
which downloads PySpark.

## Step 2. Start the Kafka server

The next two cells start the broker.

The first prepares a folder for Kafka to store messages in. The second starts the server
and waits until it answers.

Starting a server is a little different from running normal code: a server does not finish,
it keeps running. So we start it **in the background** and let it run while we carry on.

In [5]:
import subprocess, time, os, glob

LOG_DIR = '/content/kraft-logs'

# Where the config lives moved between Kafka 3 and Kafka 4, and so did the format
# command. Detect which layout we got, so this cell keeps working whichever version
# the CDN happens to be serving.
if os.path.exists(KAFKA_DIR + '/config/kraft/server.properties'):
    CONFIG = KAFKA_DIR + '/config/kraft/server.properties'   # Kafka 3.x
    FORMAT_EXTRA = []
else:
    CONFIG = KAFKA_DIR + '/config/server.properties'         # Kafka 4.x
    FORMAT_EXTRA = ['--standalone']   # 4.x refuses to format without this

print('using config:', CONFIG)

def kafka(*args):
    """Run a Kafka command line tool and return its output."""
    cmd = [KAFKA_DIR + '/bin/' + args[0]] + list(args[1:])
    return subprocess.run(cmd, capture_output=True, text=True).stdout

if not os.path.exists(LOG_DIR + '/meta.properties'):
    cluster_id = subprocess.run([KAFKA_DIR + '/bin/kafka-storage.sh', 'random-uuid'],
                                capture_output=True, text=True).stdout.strip()
    print('cluster id:', cluster_id)
    !sed -i "s#^log.dirs=.*#log.dirs={LOG_DIR}#" {CONFIG}
    subprocess.run([KAFKA_DIR + '/bin/kafka-storage.sh', 'format',
                    '-t', cluster_id, '-c', CONFIG,
                    '--ignore-formatted'] + FORMAT_EXTRA,
                   capture_output=True, text=True)
    print('storage formatted at', LOG_DIR)
else:
    print('storage already formatted')

using config: /content/kafka/config/server.properties
storage already formatted


In [6]:
# Start the broker in the background and wait until it answers.
t0 = time.time()
broker = subprocess.Popen(
    [KAFKA_DIR + '/bin/kafka-server-start.sh', CONFIG],
    stdout=open('/content/broker.log', 'w'), stderr=subprocess.STDOUT)

BOOTSTRAP = 'localhost:9092'

for attempt in range(90): #instead of waiting for a period we check availability every second
    if 'Error' not in kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP, '--list'):
        print('broker is up after about %.0f seconds' % (time.time() - t0))
        break
    time.sleep(1)
else:
    print('broker did not start. Read the log with:  !tail -40 /content/broker.log')

broker is up after about 21 seconds


You should see **`broker is up after atleast 2 seconds`**.

If it says the broker did not start, run this in a new cell to see why:

```python
!tail -40 /content/broker.log
```

You now have a real Kafka server running inside this Colab machine. Its address is
`localhost:9092`: `localhost` means "this computer", and `9092` is the door number Kafka
listens on.

## Step 3. Create a topic

A **topic** is a named mailbox. Let us make one called `clicks`.

Two settings to notice:

- **`--partitions 3`** splits the mailbox into 3 parts. Kafka can put different parts on
  different computers, which is how it handles very large volumes. Three is plenty for us.
- **`--replication-factor 1`** keeps one copy of everything. A real company uses 3, so that
  losing a computer does not lose the data. We have only one computer, so 1 is our only
  option.

In [7]:
# Create the topic.
print(kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP, #BOOTSTRAP = 'localhost:9092'
            '--create', '--topic', 'clicks',
            '--partitions', '3', '--replication-factor', '1'))

# Ask Kafka to describe it back to us.
print(kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP,
            '--describe', '--topic', 'clicks'))

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Error while executing topic command : Topic 'clicks' already exists.

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Topic: clicks	TopicId: eNoKsnndQzGxryf05VgD4g	PartitionCount: 3	ReplicationFactor: 1	Configs: min.insync.replicas=1,segment.bytes=1073741824
	Topic: clicks	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: clicks	Partition: 1	Leader: 1	Replicas: 1	Isr

You should see `Created topic clicks.` followed by a description showing
`PartitionCount: 3`.

The three lines starting `Partition: 0`, `Partition: 1` and `Partition: 2` are the three
parts of the mailbox. `Leader: 1` just means our one and only server is looking after all
of them.

You may also see a warning mentioning periods and underscores in topic names. It does not
apply to `clicks`, so ignore it.

## Step 4. Send some messages

Now we put messages in. A program that sends to Kafka is called a **producer**.

Our messages will be simple lines of text, with three things separated by commas:

```
alice,/cart,25.50
```

That is a user, a page they visited, and how much they spent. Real systems usually send
JSON instead, which the advanced notebook covers. Plain text is easier to read while you
are learning.

**Two things to know about the code below.**

- `producer.send(...)` does not actually send straight away. It puts the message in a
  queue, and the producer sends them in groups because that is faster.
- `producer.flush()` waits until Kafka has definitely received everything. **Always call it
  before you try to read the messages back**, or you may look for data that has not arrived
  yet. This catches almost everyone the first time.

In [8]:
import random
from kafka import KafkaProducer

# Connect to Kafka. This producer stays connected and can be reused.
producer = KafkaProducer(bootstrap_servers=BOOTSTRAP)

USERS = ['alice', 'bob', 'carlos', 'diana']
PAGES = ['/home', '/search', '/product', '/cart', '/checkout']

# Build 12 made up click records.
messages = []
for i in range(12):
    user = random.choice(USERS)
    page = random.choice(PAGES)
    amount = round(random.uniform(5, 120), 2)
    messages.append('%s,%s,%s' % (user, page, amount))

# Send them. .encode() turns text into bytes, which is all Kafka understands.
for m in messages:
    producer.send('clicks', value=m.encode('utf-8'))

producer.flush()          # wait until Kafka has them all

print('sent %d messages:' % len(messages))
for m in messages[:5]:
    print('   ', m)
print('    ...')

sent 12 messages:
    carlos,/product,20.41
    diana,/search,78.69
    carlos,/product,35.24
    diana,/cart,47.14
    carlos,/cart,46.14
    ...


You should see `sent 12 messages:` and the first few of them.

Nothing has read them yet. They are sitting in the `clicks` topic, waiting. They will stay
there whether or not anyone ever reads them, which is exactly the point of a queue.

## Step 5. Read the messages back

A program that reads from Kafka is called a **consumer**.

**Three settings worth understanding.**

| Setting | What it does |
|---|---|
| `auto_offset_reset='earliest'` | Start from the **first** message in the topic. The other option, `latest`, means "only messages sent from now on", which is the usual reason a beginner's consumer prints nothing. |
| `consumer_timeout_ms=10000` | Give up after 10 seconds of silence. Without this the cell would wait forever, because a consumer normally expects more messages to arrive eventually. |
| `.decode('utf-8')` | Turn the bytes back into text. Kafka stores bytes and knows nothing about what they mean, so converting back is your job. |

In [9]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'clicks',
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset='earliest',   # start at the very beginning
    consumer_timeout_ms=10000)      # stop after 10 seconds of quiet

print('reading...')
count = 0
for message in consumer:
    text = message.value.decode('utf-8')     # bytes back into text
    print('   partition %d, position %d:  %s'
          % (message.partition, message.offset, text))
    count += 1

consumer.close()
print()
print('read %d messages' % count)

reading...
   partition 1, position 0:  alice,/cart,5.8
   partition 1, position 1:  carlos,/product,13.89
   partition 1, position 2:  alice,/checkout,58.07
   partition 1, position 3:  alice,/home,39.7
   partition 1, position 4:  diana,/checkout,13.2
   partition 1, position 5:  carlos,/cart,102.13
   partition 1, position 6:  alice,/search,29.44
   partition 1, position 7:  diana,/product,30.08
   partition 1, position 8:  alice,/search,89.79
   partition 1, position 9:  carlos,/product,11.78
   partition 0, position 0:  diana,/cart,10.85
   partition 0, position 1:  bob,/home,91.81
   partition 0, position 2:  bob,/search,109.54
   partition 2, position 0:  bob,/search,113.32
   partition 2, position 1:  carlos,/product,33.98
   partition 2, position 2:  alice,/checkout,101.53
   partition 2, position 3:  carlos,/product,20.41
   partition 2, position 4:  diana,/search,78.69
   partition 2, position 5:  carlos,/product,35.24
   partition 2, position 6:  diana,/cart,47.14
   partit

You should see all 12 messages, each with a partition number and a position.

**Two things worth noticing.**

1. **The messages are spread across the three partitions.** We did not say which partition
   to use, so Kafka spread them around.

2. **The position numbers start again from 0 in each partition.** Position 0 in partition 1
   is a completely different message from position 0 in partition 2. Kafka calls this
   position an **offset**, and offsets only make sense within one partition.

Also notice: reading did **not** remove the messages. They are still in the topic. A second
consumer could read exactly the same 12 messages, and that is one of the most useful things
about Kafka.

## Step 6. Ask questions with Spark

Reading messages one at a time is fine for 12 of them. For twelve million you want Spark,
which can read the whole topic as a **table** and answer questions about it.

First we start Spark. This takes a few seconds, because it is starting a Java program
behind the scenes.

In [10]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('KafkaBasics')
         .config('spark.sql.shuffle.partitions', '4') # free tier usually gives you 2 vCPUs,
                                                      # total works divided in 4 tasks running 2 at a time
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR') # do not show too many warnings, only show ERRORs
print('Spark', spark.version, 'is ready')

Spark 4.0.1 is ready


Now read the topic into a table.

Kafka gives everything back as **bytes**, in a column called `value`. So the first job is
always to turn that into text, which is what `CAST(value AS STRING)` does.

In [11]:
raw = (spark.read.format('kafka')
       .option('kafka.bootstrap.servers', BOOTSTRAP)  #BOOTSTRAP = 'localhost:9092'
       .option('subscribe', 'clicks')  # topics is clicks
       .option('startingOffsets', 'earliest')
       .load())

print('Kafka gives us these columns:')
raw.printSchema()

print('number of messages:', raw.count())

Kafka gives us these columns:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

number of messages: 24


Those seven columns are the same for every Kafka topic you will ever read. The one you
usually care about is `value`, which holds your message. `partition` and `offset` tell you
where it was stored.

Now turn the text into proper columns. Our messages look like `alice,/cart,25.50`, so we
split on the comma and take the three pieces.

In [12]:
# Turn bytes into text, then split "alice,/cart,25.50" into three columns.
# sample value: "alice,/cart,25.50"
text = raw.selectExpr("CAST(value AS STRING) AS line") #take the value column, convert it to text, call the result line.
parts = F.split(text['line'], ',')

clicks = text.select(
    parts.getItem(0).alias('user'),
    parts.getItem(1).alias('page'),
    parts.getItem(2).cast('double').alias('amount'))

clicks.show()

+------+---------+------+
|  user|     page|amount|
+------+---------+------+
| diana|    /cart| 10.85|
|   bob|    /home| 91.81|
|   bob|  /search|109.54|
| alice|    /cart|   5.8|
|carlos| /product| 13.89|
| alice|/checkout| 58.07|
| alice|    /home|  39.7|
| diana|/checkout|  13.2|
|carlos|    /cart|102.13|
| alice|  /search| 29.44|
| diana| /product| 30.08|
| alice|  /search| 89.79|
|carlos| /product| 11.78|
|   bob|  /search|113.32|
|carlos| /product| 33.98|
| alice|/checkout|101.53|
|carlos| /product| 20.41|
| diana|  /search| 78.69|
|carlos| /product| 35.24|
| diana|    /cart| 47.14|
+------+---------+------+
only showing top 20 rows


Now it is an ordinary table with three columns, and you can ask normal questions of it.

`.cast('double')` on the last one matters: without it, `amount` would still be text, and
you cannot add up text.

### Query 1. How many clicks per user?

`groupBy` then `count` is the same idea as `GROUP BY` in SQL.

In [13]:
clicks.groupBy('user').count().orderBy('user').show()

+------+-----+
|  user|count|
+------+-----+
| alice|    6|
|   bob|    3|
|carlos|   10|
| diana|    5|
+------+-----+



### Query 2. How much did each user spend?

`agg` means aggregate. Here we ask for two numbers per user at once.

In [14]:
(clicks.groupBy('user')
       .agg(F.round(F.sum('amount'), 2).alias('total_spent'),
            F.round(F.avg('amount'), 2).alias('average'))
       .orderBy(F.desc('total_spent'))
       .show())

+------+-----------+-------+
|  user|total_spent|average|
+------+-----------+-------+
|carlos|     522.65|  52.27|
| alice|     324.33|  54.06|
|   bob|     314.67| 104.89|
| diana|     179.96|  35.99|
+------+-----------+-------+



### Query 3. Which clicks were worth more than 50?

`filter` keeps only the rows where something is true.

In [15]:
big = clicks.filter(clicks['amount'] > 50)
print('clicks over 50:', big.count())
big.orderBy(F.desc('amount')).show()

clicks over 50: 11
+------+---------+------+
|  user|     page|amount|
+------+---------+------+
|   bob|  /search|113.32|
|   bob|  /search|109.54|
|carlos|    /cart|102.13|
| alice|/checkout|101.53|
|carlos| /product| 98.78|
|   bob|    /home| 91.81|
| alice|  /search| 89.79|
|carlos|/checkout| 81.13|
|carlos|    /home| 79.17|
| diana|  /search| 78.69|
| alice|/checkout| 58.07|
+------+---------+------+



### The same thing in SQL

If you prefer SQL, Spark will take it. Register the table under a name, then write normal
SQL against it. The answer is identical; only the way you asked changed.

In [16]:
clicks.createOrReplaceTempView('clicks_table')

spark.sql('''
    SELECT page,
           COUNT(*)              AS visits,
           ROUND(SUM(amount), 2) AS revenue
    FROM clicks_table
    GROUP BY page
    ORDER BY revenue DESC
''').show()

+---------+------+-------+
|     page|visits|revenue|
+---------+------+-------+
|  /search|     5| 420.78|
|/checkout|     4| 253.93|
| /product|     7| 244.16|
|    /cart|     5| 212.06|
|    /home|     3| 210.68|
+---------+------+-------+



## Step 7. Stop everything

Good practice, even though Colab will throw this machine away shortly.

Stop things in this order: Spark first, then the producer, then the Kafka server last. If
you stopped Kafka first, the others would spend their final moments complaining that it
had vanished.

In [17]:
spark.stop()
producer.close()
broker.terminate()
broker.wait(timeout=30)
print('everything stopped')

everything stopped


---

## What you learned

| Idea | In one sentence |
|---|---|
| **Broker** | The Kafka server that stores your messages. |
| **Topic** | A named mailbox inside it. |
| **Partition** | One of the slices a topic is split into, so it can grow across computers. |
| **Offset** | A message's position inside one partition, counting from 0. |
| **Producer** | A program that sends messages. |
| **Consumer** | A program that reads them. |
| **flush()** | Waits until Kafka really has your messages. Forget it and your data looks missing. |

Three things worth remembering:

1. **Kafka stores bytes.** You call `.encode()` on the way in and `.decode()` on the way
   out, or `CAST(value AS STRING)` in Spark.
2. **Reading does not delete.** Ten different programs can read the same topic without
   getting in each other's way.
3. **Offsets are per partition**, so "message number 3" means nothing unless you also say
   which partition.

## If something went wrong

| What you saw | What to do |
|---|---|
| `NameError: name 'kafka' is not defined` | A cell was skipped. Start again from the top, in order. |
| The consumer printed nothing | Check you ran the sending cell, and that `auto_offset_reset` is `'earliest'`. |
| `broker did not start` | Run `!tail -40 /content/broker.log` to see the reason, then re-run from Step 1. |
| `Failed to find data source: kafka` | Setup cell 4 did not finish. Re-run it, then **Runtime > Restart session**, then start again from Step 1. |
| Everything is broken after a long pause | Colab recycled the machine. Run every cell again from Step 1. |

## Try these

1. Send 50 messages instead of 12. Does the spread across partitions get more even?
2. Change `auto_offset_reset` to `'latest'` and re-run Step 5. Why does it print nothing?
3. Write a query that finds each user's single largest purchase. Hint: `F.max`.
4. Run the consumer cell twice. Do you get the messages both times? Why?

## Next

`kafka_streaming_lab_advanced.ipynb` takes the same setup and adds live streaming: answers
that update by themselves as new messages arrive, and counting events in time windows.